In [31]:
#jupyter nbconvert --to python preprocesamiento.ipynb. #Convertir codigo python punta a punta

import pandas as pd
import numpy as np

#LOAD DATASETS
datafiles = {
    "Fenomenos": "../Dataset_INUMET/Fenomenos.csv",
    "Humedad": "../Dataset_INUMET/Humedad.csv",
    "Lluvia": "../Dataset_INUMET/Lluvia.csv",
    "Temperatura": "../Dataset_INUMET/Temperatura.csv",
    "Visibilidad": "../Dataset_INUMET/Visibilidad.csv",
    "VientoSalto": "../Dataset_INUMET/VientoSalto.csv",
    "VientoCarrasco": "../Dataset_INUMET/VientoCarrasco.csv"
}

dataframes = {}
for name, path in datafiles.items():
    dataframes[name] = pd.read_csv(path, sep = ";", encoding='utf-8')
# Access data like: dataframes["Temperature"]


In [32]:
# ====== 1. Lluvia.csv ======
print("Procesando datos Lluvia.csv")
df_lluvia = dataframes["Lluvia"].copy()

df_lluvia = df_lluvia.rename(columns={
    "Estación": "estacion",
    "Cod. Pluvio": "cod_pluvio",
    "Fecha": "fecha",
    "[mm]": "precip_mm",
    "Comentario": "coment_lluv"
})

# Convertir fecha y precipitación
df_lluvia["fecha"] = pd.to_datetime(df_lluvia["fecha"], format="%d/%m/%Y", errors="coerce")
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].replace("TRAZA", "0.0")  # TRAZA -> 0.0
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].astype(float)

# Criterio: día lluvioso si precipitación >= 1.0 mm
precip_minima = 1.0
df_lluvia["dia_lluvioso"] = (df_lluvia["precip_mm"] >= precip_minima).astype(int)

# Desplazar un día hacia atrás: hoy=1 si mañana llueve
# df_lluvia["lluvia_mañana"] = df_lluvia["dia_lluvioso"].shift(-1)

# Criterio: mes lluvioso si precipitación >= 100.0 mm
precip = df_lluvia["precip_mm"].values # array de precipitación

precip_30_dias = np.zeros_like(precip) # array de ceros del mismo tamaño

for i in range(len(precip)):
    precip_30_dias[i] = precip[i+1:i+31].sum() # suma de los próximos 30 días

df_lluvia["lluvia_30_dias"] = precip_30_dias

df_lluvia["mes_lluvioso"] = (df_lluvia["lluvia_30_dias"] >= 100.0).astype(int)

df_lluvia = df_lluvia[["fecha", "precip_mm", "dia_lluvioso", "mes_lluvioso"]]

# # Para mostrar todas las filas y columnas
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)

print(df_lluvia.head(60))


Procesando datos Lluvia.csv
        fecha  precip_mm  dia_lluvioso  mes_lluvioso
0  1950-01-01        4.6             1             0
1  1950-01-02        0.0             0             0
2  1950-01-03        0.0             0             0
3  1950-01-04        0.0             0             0
4  1950-01-05        0.0             0             0
5  1950-01-06        0.0             0             0
6  1950-01-07        0.0             0             0
7  1950-01-08        0.0             0             0
8  1950-01-09        0.0             0             0
9  1950-01-10        0.0             0             0
10 1950-01-11        0.0             0             0
11 1950-01-12        0.0             0             0
12 1950-01-13        0.0             0             0
13 1950-01-14        0.0             0             0
14 1950-01-15        0.0             0             0
15 1950-01-16        0.0             0             0
16 1950-01-17        0.0             0             0
17 1950-01-18     

In [33]:
# ====== 2. Humedad.csv ======
print("Procesando datos Humedad.csv")
df_hum = dataframes["Humedad"].copy()

df_hum = df_hum.rename(columns={
    "Fecha/hora": "fecha",
    "HumedadCarrasco ": "hum_carrasco",  
    "HumedadSalto ": "hum_salto"
})

# Convertir fecha
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"], errors="coerce")
df_hum["fecha"] = df_hum["fecha"].dt.date  # Extraer solo la fecha (sin hora)
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"])  # Volver a datetime

# Eliminar columnas no deseadas
df_hum = df_hum.drop(columns=["Unnamed: 3", "Unnamed: 4"], errors="ignore")  # errors="ignore" evita error si no existen

# Agrupar por fecha (en caso de múltiples mediciones por día)
df_hum = df_hum.groupby("fecha").agg({
    "hum_carrasco": "mean",
    "hum_salto": "mean"
}).reset_index()

print(df_hum.head())
print(df_hum.isna().sum())

Procesando datos Humedad.csv
       fecha  hum_carrasco  hum_salto
0 1980-01-01           NaN       69.0
1 1980-01-02          60.0       61.0
2 1980-01-03          76.0       63.0
3 1980-01-04          60.0       82.0
4 1980-01-05          66.0       84.0
fecha              0
hum_carrasco      12
hum_salto       1425
dtype: int64


In [34]:
# ====== 3. Temperatura.csv ======
print("Procesando datos Temperatura.csv")

df_temp = dataframes["Temperatura"].copy()

df_temp = df_temp.rename(columns={
    "Fecha": "fecha",
    "TmáxCarrasco": "tmax_carrasco",
    "TmínCarrasco": "tmin_carrasco",
    "TmáxSalto": "tmax_salto",
    "TmínSalto": "tmin_salto"
})

df_temp["fecha"] = pd.to_datetime(df_temp["fecha"], errors="coerce", dayfirst=True) 
# Eliminar fechas no válidas
df_temp = df_temp.dropna(subset=["fecha"]) #! existen varias fechas no definidas, nAT 
df_temp = df_temp.drop(columns=["Unnamed: 5"], errors="ignore")  

df_temp["amplitud_carrasco"] = df_temp["tmax_carrasco"] - df_temp["tmin_carrasco"]
df_temp["amplitud_salto"] = df_temp["tmax_salto"] - df_temp["tmin_salto"]
df_temp["tprom_carrasco"] = (df_temp["tmax_carrasco"] + df_temp["tmin_carrasco"]) / 2
df_temp["tprom_salto"] = (df_temp["tmax_salto"] + df_temp["tmin_salto"]) / 2

df_temp


Procesando datos Temperatura.csv


,fecha,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,tprom_carrasco,tprom_salto
0,1970-01-01,32.6,21.0,32.0,19.8,11.6,12.2,26.80,25.90
1,1970-01-02,33.8,22.0,35.4,20.7,11.8,14.7,27.90,28.05
2,1970-01-03,28.1,21.5,30.8,21.3,6.6,9.5,24.80,26.05
3,1970-01-04,24.0,18.8,34.1,20.0,5.2,14.1,21.40,27.05
4,1970-01-05,30.5,20.2,37.2,23.0,10.3,14.2,25.35,30.10
...,...,...,...,...,...,...,...,...,...
20303,2025-08-03,17.3,8.4,20.7,11.3,8.9,9.4,12.85,16.00
20304,2025-08-04,13.9,6.3,17.5,8.2,7.6,9.3,10.10,12.85
20305,2025-08-05,12.0,7.5,17.1,3.4,4.5,13.7,9.75,10.25
20306,2025-08-06,15.5,2.8,18.3,3.4,12.7,14.9,9.15,10.85


In [35]:
# ===== VientoCarrasco.csv =====
df_vientocarr = dataframes["VientoCarrasco"]

df_vientocarr = df_vientocarr.rename(columns={
  "Fecha " : "fecha",
  "DirVCarrasco" : "dir_viento_carrasco",
  "IntVCarrasco" : "int_viento_carrasco",
  "IntRCarrasco" : "int_racha_viento_carrasco"
})

df_vientocarr["fecha"] = pd.to_datetime(df_vientocarr["fecha"], dayfirst=True)
df_vientocarr["fecha"] = df_vientocarr["fecha"].dt.date
df_vientocarr["fecha"] = pd.to_datetime(df_vientocarr["fecha"])
df_vientocarr["dir_viento_carrasco"] = pd.to_numeric(df_vientocarr["dir_viento_carrasco"],errors="coerce")

df_vientocarr = df_vientocarr.groupby("fecha").agg({
    "dir_viento_carrasco": "mean",                  # promedio
    "int_viento_carrasco": ["mean", "max"],         # promedio y máximo
    "int_racha_viento_carrasco": ["mean", "max"]    # promedio y máximo
})
df_vientocarr.columns = [
    "dir_viento_carrasco_prom",
    "int_viento_carrasco_prom",
    "int_viento_carrasco_max",
    "int_racha_viento_carrasco_prom",
    "int_racha_viento_carrasco_max"
]

df_vientocarr.head()

,dir_viento_carrasco_prom,int_viento_carrasco_prom,int_viento_carrasco_max,int_racha_viento_carrasco_prom,int_racha_viento_carrasco_max
fecha,,,,,
1986-07-07,52.500000,12.687500,20.0,NaN,NaN
1986-07-08,41.666667,15.500000,25.0,NaN,NaN
1986-07-09,93.333333,9.416667,15.0,NaN,NaN
1986-07-10,62.916667,12.291667,19.0,NaN,NaN
1986-07-11,179.166667,14.333333,22.0,NaN,NaN


In [36]:
# ===== VientoSalto.csv =====
df_vientosalto = dataframes["VientoSalto"]

df_vientosalto = df_vientosalto.rename(columns={
  "Fecha/hora" : "fecha",
  "DirVSalto" : "dir_viento_salto",
  "IntVSalto" : "int_viento_salto",
  "IntRSalto" : "int_racha_viento_salto"
})

df_vientosalto["fecha"] = pd.to_datetime(df_vientosalto["fecha"], dayfirst=True)
df_vientosalto["fecha"] = df_vientosalto["fecha"].dt.date
df_vientosalto["fecha"] = pd.to_datetime(df_vientosalto["fecha"])
df_vientosalto["dir_viento_salto"] = pd.to_numeric(df_vientosalto["dir_viento_salto"],errors="coerce")

df_vientosalto = df_vientosalto.groupby("fecha").agg({
    "dir_viento_salto": "mean",                  # promedio
    "int_viento_salto": ["mean", "max"],         # promedio y máximo
    "int_racha_viento_salto": ["mean", "max"]    # promedio y máximo
})
df_vientosalto.columns = [
    "dir_viento_salto_prom",
    "int_viento_salto_prom",
    "int_viento_salto_max",
    "int_racha_viento_salto_prom",
    "int_racha_viento_salto_max"
]

df_vientosalto.head()

,dir_viento_salto_prom,int_viento_salto_prom,int_viento_salto_max,int_racha_viento_salto_prom,int_racha_viento_salto_max
fecha,,,,,
1980-01-01,151.250000,5.250,9,NaN,NaN
1980-01-02,132.500000,5.875,8,NaN,NaN
1980-01-03,233.750000,4.375,7,NaN,NaN
1980-01-04,71.428571,4.250,6,NaN,NaN
1980-01-05,123.750000,10.250,14,NaN,NaN


In [37]:
# ===== Visibilidad.csv =====

df_vis = dataframes["Visibilidad"]

df_vis = df_vis.rename(columns={
    "Fecha/hora": "fecha",
    "VisibilidadCarrasco ": "vis_carrasco",
    "VisibilidadSalto ": "vis_salto"
})

df_vis["fecha"] = pd.to_datetime(
    df_vis["fecha"],
    dayfirst=True,
    infer_datetime_format=True,  # infiere si hay hora o no
    errors = "coerce"
)
df_vis["fecha"] = df_vis["fecha"].dt.date
df_vis["fecha"] = pd.to_datetime(df_vis["fecha"])

df_vis["vis_carrasco"] = pd.to_numeric(df_vis["vis_carrasco"], errors="coerce")
df_vis["vis_salto"] = pd.to_numeric(df_vis["vis_salto"], errors="coerce")

df_vis = df_vis.groupby("fecha").agg({
    "vis_carrasco": ["mean", "max", "min"],
    "vis_salto": ["mean", "max", "min"]
})

df_vis.columns = [
    "vis_carrasco_prom", "vis_carrasco_max", "vis_carrasco_min",
    "vis_salto_prom", "vis_salto_max", "vis_salto_min"
]

df_vis.head()

C:\Users\cabre\AppData\Local\Temp\ipykernel_31780\1862289934.py:11: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_vis["fecha"] = pd.to_datetime(


,vis_carrasco_prom,vis_carrasco_max,vis_carrasco_min,vis_salto_prom,vis_salto_max,vis_salto_min
fecha,,,,,,
1995-07-03,NaN,NaN,NaN,11.260870,15.0,6.0
1995-07-04,NaN,NaN,NaN,12.956522,15.0,10.0
1995-07-05,NaN,NaN,NaN,8.565217,10.0,6.0
1995-07-06,NaN,NaN,NaN,13.347826,20.0,4.0
1995-07-07,NaN,NaN,NaN,7.652174,15.0,2.0


In [38]:
# ===== Fenomenos.csv =====

df_fen = dataframes["Fenomenos"]

df_fen = df_fen.rename(columns={
    "Fecha/hora": "fecha",
    "FenomenosCarrasco ": "fen_carrasco",
    "FenomenosSalto ": "fen_salto"
})
df_fen = df_fen.drop(columns=["Unnamed: 3", "Unnamed: 4"], axis=1)

df_fen["fecha"] = df_fen["fecha"].astype(str).str.split(" ").str[0]
df_fen["fecha"] = pd.to_datetime(df_fen["fecha"], dayfirst=True, errors="coerce")

df_carr = df_fen[["fecha", "fen_carrasco"]].copy()
df_carr["fen_carrasco"] = df_carr["fen_carrasco"].fillna(0).astype(int).astype(str)  # 0 = sin fenómeno
df_carr = pd.get_dummies(df_carr, columns=["fen_carrasco"])

df_salto = df_fen[["fecha", "fen_salto"]].copy()
df_salto["fen_salto"] = df_salto["fen_salto"].fillna(0).astype(int).astype(str)
df_salto = pd.get_dummies(df_salto, columns=["fen_salto"])

df_carr_daily = df_carr.groupby("fecha").sum().clip(upper=1).reset_index()
df_salto_daily = df_salto.groupby("fecha").sum().clip(upper=1).reset_index()

df_fen = pd.merge(df_carr_daily, df_salto_daily, on="fecha", how="outer")

df_fen.head()



,fecha,fen_carrasco_0,fen_carrasco_1,fen_carrasco_10,fen_carrasco_11,fen_carrasco_12,fen_carrasco_13,fen_carrasco_14,fen_carrasco_15,fen_carrasco_16,...,fen_salto_80,fen_salto_81,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99
0,2000-01-01,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2000-01-02,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2000-01-03,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2000-01-04,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2000-01-05,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [39]:
# Merge de dataset procesados

print("Combinando datasets en df_global")

df_global = df_lluvia.copy()
# Hacer merge con humedad
df_global = pd.merge(df_global, df_hum, on="fecha", how="outer")

# Hacer merge con temperatura
df_global = pd.merge(df_global, df_temp, on="fecha", how="outer")

"""
df_global = df_global[[ "fecha",  
                       "hum_carrasco", "hum_salto",
                       "tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto", "amplitud_carrasco", "amplitud_salto", "dia_lluvioso", "mes_lluvioso"]]
"""

df_global = pd.merge(df_global,df_vientocarr,how="left",on="fecha")
df_global = pd.merge(df_global,df_vientosalto,how="left",on="fecha")
df_global = pd.merge(df_global,df_vis,how="left",on="fecha")
df_global = pd.merge(df_global,df_fen,how="left",on="fecha")
cols = [c for c in ("dia_lluvioso", "mes_lluvioso") if c in df_global.columns]
df_global = df_global[[c for c in df_global.columns if c not in cols] + cols] #pongo dia y mes lluvioso al final

df_global.head()

Combinando datasets en df_global


,fecha,precip_mm,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,...,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99,dia_lluvioso,mes_lluvioso
0,1950-01-01,4.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0
1,1950-01-02,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,1950-01-03,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
3,1950-01-04,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
4,1950-01-05,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0


In [40]:
total_filas = len(df_global)
conteo = df_global.notna().sum().reset_index()
conteo.columns = ["columna", "no_nulos"]
conteo["total_filas"] = total_filas
conteo["porcentaje_no_nulos"] = conteo["no_nulos"] / total_filas * 100

# Guardar como archivo de texto
with open("conteo_no_nulos.txt", "w", encoding="utf-8") as f:
    f.write(conteo.to_string(index=False))

A continuación se termina de preprocesar los datos para entrenar a los modelos (Árbol de Decisión, Naive Bayes).
Como se quiere hacer predicciones climáticas, para clasificar una instancia como "día lluvioso" o "día no lluvioso" se deben mirar los días anteriores.
Como estos modelos solo usan los atributos de una instancia para clasificar a la misma, se selecciona un "tamaño de ventana" (variable 'ventana') que corresponde a la cantidad de días previos que se considerarán para la predicción. En función de esta variable, se sustituye cada fila de atributos en los datos por una combinación (ej. promedio, máximo, etc.) de los atributos de los días anteriores.
De este modo, por ejemplo, se puede predecir si mañana lloverá, considerando los atributos hasta el día de hoy.

In [41]:
import pandas as pd

def preparar_dataset(df_global, ventana):
    """
    Prepara el dataset para entrenar modelos utilizando la combinación de días anteriores.

    Parámetros:
    -----------
    df_global : pd.DataFrame
        DataFrame original con todos los datos.
    ventana : int
        Cantidad de días anteriores a considerar para el rolling.

    Retorna:
    --------
    df_modelo : pd.DataFrame
        DataFrame preparado con columnas numéricas agregadas por rolling y columnas one-hot ajustadas.
    """
    df_modelo = df_global.copy()
    
    # Eliminamos columna que no vamos a usar
    df_modelo = df_modelo.drop(columns=['precip_mm'], errors='ignore')

    # Columnas numéricas por tipo de estadística
    cols_promedio = ['hum_carrasco','hum_salto', 
                     'dir_viento_carrasco_prom','int_viento_carrasco_prom',
                     'tprom_carrasco','tprom_salto',
                     'amplitud_carrasco','amplitud_salto',
                     'int_racha_viento_carrasco_prom','int_racha_viento_salto_prom',
                     'dir_viento_salto_prom','int_viento_salto_prom',
                     'vis_carrasco_prom','vis_salto_prom']

    cols_maximo = ['tmax_carrasco','tmax_salto', 
                   'int_racha_viento_carrasco_max','int_racha_viento_carrasco_max',
                   'int_viento_salto_max','int_racha_viento_salto_max',
                   'vis_carrasco_max','vis_salto_max']

    cols_minimo = ['tmin_carrasco','tmin_salto',
                   'vis_carrasco_min','vis_salto_min']

    # Columnas one-hot
    cols_fen_carrasco = [c for c in df_modelo.columns if c.startswith('fen_carrasco_')]
    cols_fen_salto = [c for c in df_modelo.columns if c.startswith('fen_salto_')]

    # Rolling sobre días anteriores
    for col in cols_promedio:
        df_modelo[col] = df_modelo[col].rolling(window=ventana).mean().shift(1)

    for col in cols_maximo:
        df_modelo[col] = df_modelo[col].rolling(window=ventana).max().shift(1)

    for col in cols_minimo:
        df_modelo[col] = df_modelo[col].rolling(window=ventana).min().shift(1)

    for col in cols_fen_carrasco + cols_fen_salto:
        df_modelo[col] = df_modelo[col].rolling(window=ventana).sum().shift(1)

    # Drop filas con NaN en columnas de atributos o en target
    cols_features = cols_promedio + cols_maximo + cols_minimo + cols_fen_carrasco + cols_fen_salto
    df_modelo = df_modelo.dropna(how='all', subset=cols_features).reset_index(drop=True)
    
    targets = [c for c in ["dia_lluvioso", "mes_lluvioso"] if c in df_modelo.columns]
    df_modelo = df_modelo.dropna(subset=targets).reset_index(drop=True)

    # (Opcional) drop columnas con pocos datos
    # df_modelo = df_modelo.drop(columns=['int_racha_viento_carrasco_prom','int_racha_viento_salto_prom'], errors='ignore')

    return df_modelo

A continuación se procede a rellenar los valores vacíos, se utilizan dos estrategias:
1. Se utiliza la mediana de todos los datos. Obs: para las columnas de temperatura, se llena con la media del mes correspondiente, de existir.
2. Se utiliza una etiqueta especial "falta" para denotar que el dato no estaba



In [42]:
def llenar_vacios_1(df_modelo):
    """
    Llena los valores faltantes en el DataFrame de clima.
    1) Para temperaturas y humedades, usa la mediana mensual si existe.
    2) Para el resto de columnas numéricas, usa la mediana de toda la columna.
    
    Parámetros:
    -----------
    df_modelo : pd.DataFrame
        DataFrame ya preparado con columnas numéricas y columna 'dia_lluvioso'.
    
    Retorna:
    --------
    df_modelo_lleno : pd.DataFrame
        DataFrame con los valores faltantes llenados.
    """
    import pandas as pd

    df_modelo_lleno = df_modelo.copy()

    targets = [c for c in ["dia_lluvioso", "mes_lluvioso"] if c in df_modelo_lleno.columns]

    # Columnas numéricas (excepto target)
    num_cols = [col for col in df_modelo_lleno.select_dtypes(include=['float', 'int']).columns 
                if col not in targets]
    
    # Si existe columna 'fecha', llenamos con mediana mensual
    if 'fecha' in df_modelo_lleno.columns:
        df_modelo_lleno["mes"] = df_modelo_lleno["fecha"].dt.month
        
        # Columnas de temperatura y humedad
        cols_temp = [c for c in ["tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto"]
                     if c in df_modelo_lleno.columns]
        cols_hum = [c for c in ["hum_carrasco", "hum_salto"] if c in df_modelo_lleno.columns]
        
        # Mediana mensual para temperaturas
        media_temp_por_mes = df_modelo_lleno.groupby("mes")[cols_temp].mean()
        for col in cols_temp:
            df_modelo_lleno[col] = df_modelo_lleno[col].fillna(
                df_modelo_lleno["mes"].map(media_temp_por_mes[col])
            )
        
        # Mediana mensual para humedades
        media_hum_por_mes = df_modelo_lleno.groupby("mes")[cols_hum].mean()
        for col in cols_hum:
            df_modelo_lleno[col] = df_modelo_lleno[col].fillna(
                df_modelo_lleno["mes"].map(media_hum_por_mes[col])
            )
        
        # Quitar columna 'mes'
        df_modelo_lleno = df_modelo_lleno.drop(["mes"], axis=1)
    
    # Rellenar NaN restantes con mediana de toda la columna
    for col in num_cols:
        med = df_modelo_lleno[col].median()
        df_modelo_lleno[col] = df_modelo_lleno[col].fillna(med)
    
    return df_modelo_lleno

df_modelo1_vent1 = llenar_vacios_1(preparar_dataset(df_global, 1))
df_modelo1_vent2 = llenar_vacios_1(preparar_dataset(df_global, 2))
df_modelo1_vent3 = llenar_vacios_1(preparar_dataset(df_global, 3))
df_modelo1_vent5 = llenar_vacios_1(preparar_dataset(df_global, 5))
df_modelo1_vent7 = llenar_vacios_1(preparar_dataset(df_global, 7))
df_modelo1_vent10 = llenar_vacios_1(preparar_dataset(df_global, 10))
df_modelo1_vent11 = llenar_vacios_1(preparar_dataset(df_global, 11))
df_modelo1_vent20 = llenar_vacios_1(preparar_dataset(df_global, 20))
df_modelo1_vent30 = llenar_vacios_1(preparar_dataset(df_global, 30))

C:\Users\cabre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\cabre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\cabre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\cabre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarnin

In [43]:
def llenar_vacios_2(df_modelo):
    """
    Llena los valores faltantes en el DataFrame con la etiqueta -99.

    Parámetros:
    -----------
    df_modelo : pd.DataFrame
        DataFrame con columnas numéricas y target 'dia_lluvioso'.

    Retorna:
    --------
    df_modelo_lleno : pd.DataFrame
        DataFrame con valores faltantes reemplazados por -99.
    """
    df_modelo_lleno = df_modelo.copy()
    
    # Columnas numéricas (excepto target)
    targets = [c for c in ["dia_lluvioso", "mes_lluvioso"] if c in df_modelo_lleno.columns]
    num_cols = [col for col in df_modelo_lleno.select_dtypes(include=['float', 'int']).columns
                if col not in targets]

    # Llenar NaN con -99
    for col in num_cols:
        df_modelo_lleno[col] = df_modelo_lleno[col].fillna(-99)
    
    return df_modelo_lleno

df_modelo2_vent1 = llenar_vacios_2(preparar_dataset(df_global, 1))
df_modelo2_vent2 = llenar_vacios_2(preparar_dataset(df_global, 2))
df_modelo2_vent3 = llenar_vacios_2(preparar_dataset(df_global, 3))
df_modelo2_vent5 = llenar_vacios_2(preparar_dataset(df_global, 5))
df_modelo2_vent7 = llenar_vacios_2(preparar_dataset(df_global, 7))
df_modelo2_vent10 = llenar_vacios_2(preparar_dataset(df_global, 10))
df_modelo2_vent11 = llenar_vacios_2(preparar_dataset(df_global, 11))
df_modelo2_vent20 = llenar_vacios_2(preparar_dataset(df_global, 20))
df_modelo2_vent30 = llenar_vacios_2(preparar_dataset(df_global, 30))

A continuación se aplican criterios de discretización para las variables continuas (necesario para entrenar a los modelos).

In [44]:
#==================DISCRETIZACIÓN DE VARIABLES CONTINUAS============================

def discretizar(df):
    """
    Discretiza un DataFrame de clima:
    - Temperaturas y amplitudes en categorías según percentiles.
    - Dirección del viento en puntos cardinales.
    - Variables binarias fen_* y dia_lluvioso en 'si', 'no', 'falta'.
    - Resto de variables continuas en percentiles (muy_bajo, bajo, medio, alto, muy_alto).
    - Ignora -99 (se considera 'falta').
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame con columnas de clima ya llenadas (puede tener -99).
    
    Retorna:
    --------
    df_discretizado : pd.DataFrame
        DataFrame con todas las columnas discretizadas/categorizadas.
    """
    
    df_discr = df.copy()
    
    # --- Helper para percentiles ignorando -99 ---
    def percentiles_columna(col, percentiles):
        valid = col[col != -99]
        if len(valid) == 0:
            return None  # ninguna fila válida
        return np.percentile(valid, percentiles)
    
    # --- Categorizar temperaturas y amplitudes ---
    temp_cols = ["tmax_carrasco", "tmin_carrasco", "tprom_carrasco",
                 "tmax_salto", "tmin_salto", "tprom_salto"]
    amp_cols = ["amplitud_carrasco", "amplitud_salto"]
    
    percentiles_temp = {col: percentiles_columna(df_discr[col], [10, 30, 70, 90]) for col in temp_cols}
    percentiles_amp = {col: percentiles_columna(df_discr[col], [25, 50, 85]) for col in amp_cols}
    
    def categorizar_tmin_tmax_tprom(temp, p10, p30, p70, p90):
        if temp == -99:
            return "falta"
        elif temp < p10:
            return "muy_frio"
        elif temp < p30:
            return "fresco"
        elif temp < p70:
            return "agradable"
        elif temp < p90:
            return "calido"
        else:
            return "muy_calido"
    
    def categorizar_amplitud(val, p25, p50, p85):
        if val == -99:
            return "falta"
        elif val < p25:
            return "variacion_baja"
        elif val < p50:
            return "variacion_moderada"
        elif val < p85:
            return "variacion_alta"
        else:
            return "variacion_muy_alta"
    
    # Aplicar discretización temperaturas y amplitudes
    for col in temp_cols:
        pts = percentiles_temp[col]
        if pts is None:
            df_discr[col] = "falta"
        else:
            p10, p30, p70, p90 = pts
            df_discr[col] = df_discr[col].apply(lambda t: categorizar_tmin_tmax_tprom(t, p10, p30, p70, p90))
    
    for col in amp_cols:
        pts = percentiles_amp[col]
        if pts is None:
            df_discr[col] = "falta"
        else:
            p25, p50, p85 = pts
            df_discr[col] = df_discr[col].apply(lambda t: categorizar_amplitud(t, p25, p50, p85))
    
    # --- Discretizar resto de columnas ---
    def percentiles_col(col, pts):
        valid = col[col != -99]
        if len(valid) == 0:
            return None
        return np.percentile(valid, pts)
    
    def categorizar_percentiles(val, pts, etiquetas):
        if val == -99:
            return "falta"
        for i, p in enumerate(pts):
            if val < p:
                return etiquetas[i]
        return etiquetas[-1]
    
    def angulo_a_cardinal(angulo):
        if angulo == -99:
            return "falta"
        cardinales = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
        index = int(((angulo + 22.5) % 360) // 45)
        return cardinales[index]
    
    def categorizar_binario(val):
        if val == -99:
            return "falta"
        elif val == 0:
            return "no"
        elif val == 1:
            return "si"
        else:
            return "falta"
    
    temp_amp_cols = temp_cols + amp_cols
    
    for col in df_discr.columns:
        if col == "fecha":
            continue
        elif col.startswith("fen_") or col in ["dia_lluvioso", "mes_lluvioso"]:
            df_discr[col] = df_discr[col].apply(categorizar_binario)
        elif col in temp_amp_cols:
            continue
        elif col.startswith("dir_"):
            df_discr[col] = df_discr[col].apply(angulo_a_cardinal)
        else:
            pts = percentiles_col(df_discr[col], [10, 30, 70, 90])
            if pts is None:
                df_discr[col] = "falta"
                continue
            etiquetas = ["muy_bajo", "bajo", "medio", "alto", "muy_alto"]
            df_discr[col] = df_discr[col].apply(lambda x: categorizar_percentiles(x, pts, etiquetas))
    
    return df_discr


df_modelo1_vent1.to_csv('../Dataset_INUMET/data_dia_c.csv', index=False)
df_modelo1_vent1 = discretizar(df_modelo1_vent1)
df_modelo1_vent2 = discretizar(df_modelo1_vent2)
df_modelo1_vent3 = discretizar(df_modelo1_vent3)
df_modelo1_vent5 = discretizar(df_modelo1_vent5)
df_modelo1_vent7 = discretizar(df_modelo1_vent7)
df_modelo1_vent10 = discretizar(df_modelo1_vent10)
df_modelo1_vent11 = discretizar(df_modelo1_vent11)
df_modelo1_vent20 = discretizar(df_modelo1_vent20)
df_modelo1_vent30 = discretizar(df_modelo1_vent30)
df_modelo2_vent1 = discretizar(df_modelo2_vent1)
df_modelo2_vent2 = discretizar(df_modelo2_vent2)
df_modelo2_vent3 = discretizar(df_modelo2_vent3)
df_modelo2_vent5 = discretizar(df_modelo2_vent5)
df_modelo2_vent7 = discretizar(df_modelo2_vent7)
df_modelo2_vent10 = discretizar(df_modelo2_vent10)
df_modelo2_vent11 = discretizar(df_modelo2_vent11)
df_modelo2_vent20 = discretizar(df_modelo2_vent20)
df_modelo2_vent30 = discretizar(df_modelo2_vent30)  

In [45]:
from astral import moon
import pandas as pd

def calcular_fases_lunares(df, col_fecha='fecha'):
    df = df.copy()
    df[col_fecha] = pd.to_datetime(df[col_fecha])
    
    def fase_lunar(fecha):
        dia = moon.phase(fecha)  # día del ciclo lunar
        # Clasificación simplificada en 4 fases
        if dia < 1 or dia > 28:
            return "luna_nueva"
        elif dia < 7:
            return "luna_creciente"
        elif dia < 15:
            return "luna_llena"
        elif dia < 22:
            return "luna_menguante"
        else:
            return "luna_nueva"
    
    df['fase_lunar'] = df[col_fecha].apply(fase_lunar)
    return df

df_modelo1_vent1 = calcular_fases_lunares(df_modelo1_vent1)
df_modelo1_vent2 = calcular_fases_lunares(df_modelo1_vent2)
df_modelo1_vent3 = calcular_fases_lunares(df_modelo1_vent3)
df_modelo1_vent5 = calcular_fases_lunares(df_modelo1_vent5)
df_modelo1_vent7 = calcular_fases_lunares(df_modelo1_vent7)
df_modelo1_vent10 = calcular_fases_lunares(df_modelo1_vent10)
df_modelo1_vent20 = calcular_fases_lunares(df_modelo1_vent20)
df_modelo1_vent30 = calcular_fases_lunares(df_modelo1_vent30)
df_modelo2_vent1 = calcular_fases_lunares(df_modelo2_vent1)
df_modelo2_vent2 = calcular_fases_lunares(df_modelo2_vent2)
df_modelo2_vent3 = calcular_fases_lunares(df_modelo2_vent3)
df_modelo2_vent5 = calcular_fases_lunares(df_modelo2_vent5)
df_modelo2_vent7 = calcular_fases_lunares(df_modelo2_vent7)
df_modelo2_vent10 = calcular_fases_lunares(df_modelo2_vent10)
df_modelo2_vent11 = calcular_fases_lunares(df_modelo2_vent11)
df_modelo2_vent20 = calcular_fases_lunares(df_modelo2_vent20)
df_modelo2_vent30 = calcular_fases_lunares(df_modelo2_vent30)


df_modelo1_vent1.to_csv('../Dataset_INUMET/datadia1.csv', index=False)
df_modelo1_vent1.to_csv('../Datos_preprocesados/modelo1_vent1.csv', index=False)
df_modelo1_vent2.to_csv('../Datos_preprocesados/modelo1_vent2.csv', index=False)
df_modelo1_vent3.to_csv('../Datos_preprocesados/modelo1_vent3.csv', index=False)
df_modelo1_vent5.to_csv('../Datos_preprocesados/modelo1_vent5.csv', index=False)
df_modelo1_vent7.to_csv('../Datos_preprocesados/modelo1_vent7.csv', index=False)
df_modelo1_vent10.to_csv('../Datos_preprocesados/modelo1_vent10.csv', index=False)
df_modelo1_vent11.to_csv('../Datos_preprocesados/modelo1_vent11.csv', index=False)
df_modelo1_vent20.to_csv('../Datos_preprocesados/modelo1_vent20.csv', index=False)
df_modelo1_vent30.to_csv('../Datos_preprocesados/modelo1_vent30.csv', index=False)
df_modelo2_vent1.to_csv('../Datos_preprocesados/modelo2_vent1.csv', index=False)
df_modelo2_vent2.to_csv('../Datos_preprocesados/modelo2_vent2.csv', index=False)
df_modelo2_vent3.to_csv('../Datos_preprocesados/modelo2_vent3.csv', index=False)
df_modelo2_vent5.to_csv('../Datos_preprocesados/modelo2_vent5.csv', index=False)
df_modelo2_vent7.to_csv('../Datos_preprocesados/modelo2_vent7.csv', index=False)
df_modelo2_vent10.to_csv('../Datos_preprocesados/modelo2_vent10.csv', index=False)
df_modelo2_vent11.to_csv('../Datos_preprocesados/modelo2_vent11.csv', index=False)
df_modelo2_vent20.to_csv('../Datos_preprocesados/modelo2_vent20.csv', index=False)
df_modelo2_vent30.to_csv('../Datos_preprocesados/modelo2_vent30.csv', index=False)